# Experimento final · Conv + BiLSTM conjunta con 20 épocas

Misma configuración que el candidato final (`class_weight_power = 0.25`, ~62 mil instancias de train), con
**20 épocas en lugar de 10**.

En el candidato final la mejor época fue la última (10/10), la loss de train (0.145) y la de test (0.148) iban
juntas y cada época sumaba entre +0.004 y +0.009 de F1. Extrapolando las curvas, con 20 épocas el F1 promedio en
test podría llegar a ~0.81 (estaba en 0.766). Si empieza a sobreajustar antes, el pipeline se queda con la mejor
época en test.

Tiempo estimado: ~6–7 minutos por época, unas 2–2,5 horas.

**En Colab**: Entorno de ejecución → Cambiar tipo de entorno de ejecución → **GPU**. Se puede correr en paralelo con el otro notebook del experimento final (cada uno entrena un modelo); los dos guardan en `MyDrive/tp2-models` y cada uno reconoce su corrida por el experimento registrado en su `metrics.json`. Con Colab gratis puede que solo haya una GPU disponible a la vez.

## 1. Setup

In [ ]:
import json
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
BRANCH = "feat/birnn-cabeza-no-lineal"  # rama con la cabeza no lineal de la BiRNN (src/problem/heads.py)
REPO = "/content/TP2-Aprendizaje-Automatico-UBA"

if IN_COLAB:
    # Los resultados van a Drive: la máquina de Colab se borra al cerrar la sesión
    from google.colab import drive
    drive.mount("/content/drive")
    if not Path(REPO).exists():
        !git clone https://github.com/SebasAndres/TP2-Aprendizaje-Automatico-UBA.git {REPO}
    %cd {REPO}
    !git fetch origin && git checkout {BRANCH} && git pull
    !pip install -q uv && uv sync
    !mkdir -p /content/drive/MyDrive/tp2-models
    !if [ ! -L models ]; then rm -rf models && ln -s /content/drive/MyDrive/tp2-models models; fi
else:
    # Local: el notebook corre desde notebooks/experiments/, subimos hasta la raíz del repo
    %cd ../..

!nvidia-smi -L || echo "Sin GPU: Entorno de ejecución → Cambiar tipo de entorno → GPU"

## 2. Configuración

In [ ]:
EXPERIMENT = "joint_conv_bilstm"
EPOCHS = 20  # candidato final: 10
CONFIG_KEYS = ("n_epochs", "class_weight_power", "experiments")
# F1 macro en control del candidato final (promedio de las tareas del problema)
BASELINE_CONTROL = {"CapitalizationProblem": 0.7577, "PunctuationProblem": 0.7723}
CLASS_WEIGHT_POWER = 0.25  # igual que el candidato final (src/constants.py: 0.5)
PUNCTUATION_BIRNN_WEIGHT_DECAY = 0.1  # igual que el candidato final (src/problem/punctuation: 1.0)
SIZE = 1_000_000  # máximo de instancias por fuente, igual que las corridas anteriores (~62 mil de train)

# Pisamos los valores antes de importar el pipeline, sin tocar el código del repo.
# run_config los lee de los mismos lugares, así que quedan registrados en metrics.json.
OVERRIDES = (
    "import runpy, sys; "
    "import src.constants as c; "
    f"c.CLASS_WEIGHT_POWER = {CLASS_WEIGHT_POWER}; "
    "from src.problem import CapitalizationProblem, PunctuationProblem; "
    f"PunctuationProblem.bidirectional_rnn_weight_decay = {PUNCTUATION_BIRNN_WEIGHT_DECAY}; "

    "sys.argv = [\"src.pipeline\"] + sys.argv[1:]; "
    "runpy.run_module(\"src.pipeline\", run_name=\"__main__\")"
)
PIPELINE = f"MPLBACKEND=Agg uv run python -c '{OVERRIDES}'"

## 3. Entrenamiento

El `.pkl` se guarda en Drive (`MyDrive/tp2-models/<fecha>__<hora>/`) al terminar cada problema.

In [ ]:
before = set(Path("models").iterdir())
!{PIPELINE} --experiments {EXPERIMENT} --epochs {EPOCHS} \
    --capitalization-size {SIZE} --punctuation-size {SIZE}


def is_this_run(run_dir: Path) -> bool:
    """La corrida de este notebook: el otro notebook puede estar escribiendo en la misma carpeta de Drive."""
    path = run_dir / "metrics.json"
    return path.exists() and EXPERIMENT in (json.loads(path.read_text())["config"]["experiments"] or [])


new_runs = sorted(d for d in set(Path("models").iterdir()) - before if is_this_run(d))
RUN_DIR = new_runs[-1] if new_runs else None
print("Resultados en:", RUN_DIR)

## 4. Resultados

F1 macro sobre **control** (promedio de las tareas de cada problema) contra el candidato final (`models/precandidato_final`). Si `Δ` es positivo, mejora.

In [ ]:
def task_scores(run_dir: Path) -> list[tuple[str, int, dict[str, float]]]:
    """(problema, mejor época, F1 control por tarea) del modelo de este notebook."""
    report = json.loads((run_dir / "metrics.json").read_text())
    print(run_dir.name, "config:", {k: report["config"].get(k) for k in CONFIG_KEYS})
    return [
        (problem, models[EXPERIMENT]["best_epoch"], {t: v["f1_macro"] for t, v in models[EXPERIMENT]["control"].items()})
        for problem, models in report["problems"].items()
        if models.get(EXPERIMENT, {}).get("control")
    ]


if RUN_DIR is None:
    print("No se encontró la corrida: completar RUN_DIR a mano, ej: RUN_DIR = Path('models/2026-11-18__10-00-00')")
else:
    print(f"{'problema':22} {'época':>5} {'control por tarea':34} {'control':>7} {'candidato final':>15} {'Δ':>7}")
    for problem, epoch, control in task_scores(RUN_DIR):
        mean = sum(control.values()) / len(control)
        base = BASELINE_CONTROL[problem]
        tasks = " ".join(f"{t}={v:.3f}" for t, v in control.items())
        print(f"{problem:22} {epoch:5} {tasks:34} {mean:7.3f} {base:15.3f} {mean - base:+7.3f}")